### Overview

Zo všetkých variantov sa ako najlepší ukázal model Decision Tree s výberom hyperparametrov.

Naopak najhoršie výsledky dosiahol model s vyvažovaním tried pomocou technológie SMOTE


In [4]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE


In [2]:
TW_500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=500/Twitter-Relative-Sigma-500.data",
    sep=",",
    header=None
)

TW_1000= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1000/Twitter-Relative-Sigma-1000.data",
    sep=",",
    header=None
)
TW_1500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1500/Twitter-Relative-Sigma-1500.data",
    sep=",",
    header=None
)
groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label") 

TW_500.columns = columns
TW_1000.columns = columns
TW_1500.columns = columns


### 500


### 1 Baseline Decision Tree

In [5]:
X = TW_500.drop("label", axis=1)
y = TW_500["label"]

X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,random_state=42, stratify=y)

tree_baseline = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)

y_pred_tree = tree_baseline.predict(X_test)
y_prob_tree = tree_baseline.predict_proba(X_test)[:,1]


accuracy_tree = accuracy_score(y_test, y_pred_tree)
precision_tree = precision_score(y_test, y_pred_tree)
recall_tree = recall_score(y_test, y_pred_tree)
f1_tree = f1_score(y_test, y_pred_tree)
roc_auc_tree = roc_auc_score(y_test, y_prob_tree)

cm_tree = confusion_matrix(y_test, y_pred_tree)

print('1️.Baseline Decision Tree')
print("Accuracy:", accuracy_tree)
print("Precision:", precision_tree)
print("Recall:", recall_tree)
print("F1-score:", f1_tree)
print("ROC-AUC:", roc_auc_tree)

print("\nConfusion Matrix:")
p1️.Baseline Decision Tree
Accuracy: 0.972745362802928
Precision: 0.472258064516129
Recall: 0.505524861878453
F1-score: 0.4883255503669113
ROC-AUC: 0.745303827102331

Confusion Matrix:
[[27009   409]
 [  358   366]]rint(cm_tree)

1️.Baseline Decision Tree
Accuracy: 0.972745362802928
Precision: 0.472258064516129
Recall: 0.505524861878453
F1-score: 0.4883255503669113
ROC-AUC: 0.745303827102331

Confusion Matrix:
[[27009   409]
 [  358   366]]


### ️2.Balanced Decision Tree

In [6]:
tree_balanced = DecisionTreeClassifier(random_state=42,class_weight="balanced").fit(X_train, y_train)

y_pred_tree_bal = tree_balanced.predict(X_test)
y_prob_tree_bal = tree_balanced.predict_proba(X_test)[:,1]

accuracy_tree_bal = accuracy_score(y_test, y_pred_tree_bal)
precision_tree_bal = precision_score(y_test, y_pred_tree_bal)
recall_tree_bal = recall_score(y_test, y_pred_tree_bal)
f1_tree_bal = f1_score(y_test, y_pred_tree_bal)
roc_auc_tree_bal = roc_auc_score(y_test, y_prob_tree_bal)

cm_tree_bal = confusion_matrix(y_test, y_pred_tree_bal)

print('2.Balanced Decision Tree')
print("Accuracy:", accuracy_tree_bal)
print("Precision:", precision_tree_bal)
print("Recall:", recall_tree_bal)
print("F1-score:", f1_tree_bal)
print("ROC-AUC:", roc_auc_tree_bal)

print("\nConfusion Matrix:")
print(cm_tree_bal)

2.Balanced Decision Tree
Accuracy: 0.9723544879539479
Precision: 0.46120689655172414
Recall: 0.44337016574585636
F1-score: 0.45211267605633804
ROC-AUC: 0.7148465096728407

Confusion Matrix:
[[27043   375]
 [  403   321]]


### 3.Decision Tree with Stratified K-Fold Cross-Validation

In [7]:
tree_model = DecisionTreeClassifier(random_state=42)

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results_tree = cross_validate( tree_model,X,y,cv=cv,scoring=scoring)

print('3.Decision Tree with Stratified K-Fold Cross-Validation')
print("Accuracy:", cv_results_tree["test_accuracy"].mean())
print("Precision:", cv_results_tree["test_precision"].mean())
print("Recall:", cv_results_tree["test_recall"].mean())
print("F1-score:", cv_results_tree["test_f1"].mean())
print("ROC-AUC:", cv_results_tree["test_roc_auc"].mean())

3.Decision Tree with Stratified K-Fold Cross-Validation
Accuracy: 0.971373145361024
Precision: 0.4477778229264164
Recall: 0.48232044198895024
F1-score: 0.46434538385950797
ROC-AUC: 0.733303897958919


### 4.Decision Tree with Grid Search

In [8]:
tree = DecisionTreeClassifier(random_state=42)

param_grid = {
    "max_depth": [3, 5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "criterion": ["gini", "entropy"]
}

grid_tree = GridSearchCV(tree, param_grid,cv=5,scoring="f1", n_jobs=-1).fit(X_train, y_train)
best_tree = grid_tree.best_estimator_


y_pred_tree_grid = best_tree.predict(X_test)
y_prob_tree_grid = best_tree.predict_proba(X_test)[:,1]

accuracy_tree_grid = accuracy_score(y_test, y_pred_tree_grid)
precision_tree_grid = precision_score(y_test, y_pred_tree_grid)
recall_tree_grid = recall_score(y_test, y_pred_tree_grid)
f1_tree_grid = f1_score(y_test, y_pred_tree_grid)
roc_auc_tree_grid = roc_auc_score(y_test, y_prob_tree_grid)

cm_tree_grid = confusion_matrix(y_test, y_pred_tree_grid)

print('4.Decision Tree with Grid Search')
print("Best parameters:", grid_tree.best_params_)
print("Accuracy:", accuracy_tree_grid)
print("Precision:", precision_tree_grid)
print("Recall:", recall_tree_grid)
print("F1-score:", f1_tree_grid)
print("ROC-AUC:", roc_auc_tree_grid)

print("\nConfusion Matrix:")
print(cm_tree_grid)

4.Decision Tree with Grid Search
Best parameters: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 2}
Accuracy: 0.9785374173832706
Precision: 0.6271186440677966
Recall: 0.4088397790055249
F1-score: 0.49498327759197325
ROC-AUC: 0.8862854593244185

Confusion Matrix:
[[27242   176]
 [  428   296]]


### 5.Decision Tree with Grid Search with SMOTE

In [9]:
pipeline = Pipeline([("smote", SMOTE(random_state=42)),("tree", DecisionTreeClassifier(random_state=42))])


param_grid = {
    "tree__max_depth": [3, 5, 10, 20, None],
    "tree__min_samples_split": [2, 5, 10],
    "tree__min_samples_leaf": [1, 2, 5],
    "tree__criterion": ["gini", "entropy"]
}

grid_tree_smote = GridSearchCV(pipeline,param_grid,cv=5, scoring="f1",n_jobs=-1).fit(X_train, y_train)


best_tree_smote = grid_tree_smote.best_estimator_
y_pred_tree_smote = best_tree_smote.predict(X_test)
y_prob_tree_smote = best_tree_smote.predict_proba(X_test)[:,1]

accuracy_tree_smote = accuracy_score(y_test, y_pred_tree_smote)
precision_tree_smote = precision_score(y_test, y_pred_tree_smote)
recall_tree_smote = recall_score(y_test, y_pred_tree_smote)
f1_tree_smote = f1_score(y_test, y_pred_tree_smote)
roc_auc_tree_smote = roc_auc_score(y_test, y_prob_tree_smote)

cm_tree_smote = confusion_matrix(y_test, y_pred_tree_smote)

print('Decision Tree with Grid Search with SMOTE')
print("Best parameters:", grid_tree_smote.best_params_)
print("Accuracy:", accuracy_tree_smote)
print("Precision:", precision_tree_smote)
print("Recall:", recall_tree_smote)
print("F1-score:", f1_tree_smote)
print("ROC-AUC:", roc_auc_tree_smote)

print("\nConfusion Matrix:")
print(cm_tree_smote)

Decision Tree with Grid Search with SMOTE
Best parameters: {'tree__criterion': 'entropy', 'tree__max_depth': None, 'tree__min_samples_leaf': 2, 'tree__min_samples_split': 2}
Accuracy: 0.9539833700518797
Precision: 0.30431802604523644
Recall: 0.6132596685082873
F1-score: 0.4067796610169492
ROC-AUC: 0.8064287071565278

Confusion Matrix:
[[26403  1015]
 [  280   444]]


In [1]:
import pandas as pd

tree_500 = [
    {"Model": "Tree baseline", "Accuracy": 0.97275, "Precision": 0.4723, "Recall": 0.5055, "F1": 0.4883, "ROC-AUC": 0.7453},
    {"Model": "Tree balanced", "Accuracy": 0.97235, "Precision": 0.4612, "Recall": 0.4434, "F1": 0.4521, "ROC-AUC": 0.7148},
    {"Model": "Tree CV", "Accuracy": 0.97137, "Precision": 0.4478, "Recall": 0.4823, "F1": 0.4643, "ROC-AUC": 0.7333},
    {"Model": "Tree tuned", "Accuracy": 0.97854, "Precision": 0.6271, "Recall": 0.4088, "F1": 0.4950, "ROC-AUC": 0.8863},
    {"Model": "Tree + SMOTE", "Accuracy": 0.95398, "Precision": 0.3043, "Recall": 0.6133, "F1": 0.4068, "ROC-AUC": 0.8064}
]

df_tree_500 = pd.DataFrame(tree_500)
df_tree_500.sort_values(by="F1", ascending=False).round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
3,Tree tuned,0.9785,0.6271,0.4088,0.4950,0.8863
0,Tree baseline,0.9728,0.4723,0.5055,0.4883,0.7453
2,Tree CV,0.9714,0.4478,0.4823,0.4643,0.7333
1,Tree balanced,0.9724,0.4612,0.4434,0.4521,0.7148
4,Tree + SMOTE,0.9540,0.3043,0.6133,0.4068,0.8064


### 1000

### 1 Baseline Decision Tree

In [13]:
X = TW_1000.drop("label", axis=1)
y = TW_1000["label"]

X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,random_state=42, stratify=y)

tree_baseline = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)

y_pred_tree = tree_baseline.predict(X_test)
y_prob_tree = tree_baseline.predict_proba(X_test)[:,1]


accuracy_tree = accuracy_score(y_test, y_pred_tree)
precision_tree = precision_score(y_test, y_pred_tree)
recall_tree = recall_score(y_test, y_pred_tree)
f1_tree = f1_score(y_test, y_pred_tree)
roc_auc_tree = roc_auc_score(y_test, y_prob_tree)

cm_tree = confusion_matrix(y_test, y_pred_tree)

print('1️.Baseline Decision Tree')
print("Accuracy:", accuracy_tree)
print("Precision:", precision_tree)
print("Recall:", recall_tree)
print("F1-score:", f1_tree)
print("ROC-AUC:", roc_auc_tree)

print("\nConfusion Matrix:")
print(cm_tree)

1️.Baseline Decision Tree
Accuracy: 0.9921825030203966
Precision: 0.5316455696202531
Recall: 0.5361702127659574
F1-score: 0.5338983050847458
ROC-AUC: 0.766096358040269

Confusion Matrix:
[[27796   111]
 [  109   126]]


### ️2.Balanced Decision Tree

In [15]:
tree_balanced = DecisionTreeClassifier(random_state=42,class_weight="balanced").fit(X_train, y_train)

y_pred_tree_bal = tree_balanced.predict(X_test)
y_prob_tree_bal = tree_balanced.predict_proba(X_test)[:,1]

accuracy_tree_bal = accuracy_score(y_test, y_pred_tree_bal)
precision_tree_bal = precision_score(y_test, y_pred_tree_bal)
recall_tree_bal = recall_score(y_test, y_pred_tree_bal)
f1_tree_bal = f1_score(y_test, y_pred_tree_bal)
roc_auc_tree_bal = roc_auc_score(y_test, y_prob_tree_bal)

cm_tree_bal = confusion_matrix(y_test, y_pred_tree_bal)

print('2.Balanced Decision Tree')
print("Accuracy:", accuracy_tree_bal)
print("Precision:", precision_tree_bal)
print("Recall:", recall_tree_bal)
print("F1-score:", f1_tree_bal)
print("ROC-AUC:", roc_auc_tree_bal)

print("\nConfusion Matrix:")
print(cm_tree_bal)

2.Balanced Decision Tree
Accuracy: 0.9920048326344965
Precision: 0.5242718446601942
Recall: 0.4595744680851064
F1-score: 0.4897959183673469
ROC-AUC: 0.7280314021724131

Confusion Matrix:
[[27809    98]
 [  127   108]]


### 3.Decision Tree with Stratified K-Fold Cross-Validation

In [17]:
tree_model = DecisionTreeClassifier(random_state=42)

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results_tree = cross_validate( tree_model,X,y,cv=cv,scoring=scoring)

print('3.Decision Tree with Stratified K-Fold Cross-Validation')
print("Accuracy:", cv_results_tree["test_accuracy"].mean())
print("Precision:", cv_results_tree["test_precision"].mean())
print("Recall:", cv_results_tree["test_recall"].mean())
print("F1-score:", cv_results_tree["test_f1"].mean())
print("ROC-AUC:", cv_results_tree["test_roc_auc"].mean())

3.Decision Tree with Stratified K-Fold Cross-Validation
Accuracy: 0.9922036608289467
Precision: 0.5342348531519678
Recall: 0.5513992066354129
F1-score: 0.5423998082602941
ROC-AUC: 0.7736606152864587


### 4.Decision Tree with Grid Search

In [19]:
tree = DecisionTreeClassifier(random_state=42)

param_grid = {
    "max_depth": [3, 5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "criterion": ["gini", "entropy"]
}

grid_tree = GridSearchCV(tree, param_grid,cv=5,scoring="f1", n_jobs=-1).fit(X_train, y_train)
best_tree = grid_tree.best_estimator_


y_pred_tree_grid = best_tree.predict(X_test)
y_prob_tree_grid = best_tree.predict_proba(X_test)[:,1]

accuracy_tree_grid = accuracy_score(y_test, y_pred_tree_grid)
precision_tree_grid = precision_score(y_test, y_pred_tree_grid)
recall_tree_grid = recall_score(y_test, y_pred_tree_grid)
f1_tree_grid = f1_score(y_test, y_pred_tree_grid)
roc_auc_tree_grid = roc_auc_score(y_test, y_prob_tree_grid)

cm_tree_grid = confusion_matrix(y_test, y_pred_tree_grid)

print('4.Decision Tree with Grid Search')
print("Best parameters:", grid_tree.best_params_)
print("Accuracy:", accuracy_tree_grid)
print("Precision:", precision_tree_grid)
print("Recall:", recall_tree_grid)
print("F1-score:", f1_tree_grid)
print("ROC-AUC:", roc_auc_tree_grid)

print("\nConfusion Matrix:")
print(cm_tree_grid)

4.Decision Tree with Grid Search
Best parameters: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 2}
Accuracy: 0.9933195934901571
Precision: 0.6230366492146597
Recall: 0.5063829787234042
F1-score: 0.5586854460093896
ROC-AUC: 0.8868359879203648

Confusion Matrix:
[[27835    72]
 [  116   119]]


### 5.Decision Tree with Grid Search with SMOTE

In [21]:
pipeline = Pipeline([("smote", SMOTE(random_state=42)),("tree", DecisionTreeClassifier(random_state=42))])


param_grid = {
    "tree__max_depth": [3, 5, 10, 20, None],
    "tree__min_samples_split": [2, 5, 10],
    "tree__min_samples_leaf": [1, 2, 5],
    "tree__criterion": ["gini", "entropy"]
}

grid_tree_smote = GridSearchCV(pipeline,param_grid,cv=5, scoring="f1",n_jobs=-1).fit(X_train, y_train)


best_tree_smote = grid_tree_smote.best_estimator_
y_pred_tree_smote = best_tree_smote.predict(X_test)
y_prob_tree_smote = best_tree_smote.predict_proba(X_test)[:,1]

accuracy_tree_smote = accuracy_score(y_test, y_pred_tree_smote)
precision_tree_smote = precision_score(y_test, y_pred_tree_smote)
recall_tree_smote = recall_score(y_test, y_pred_tree_smote)
f1_tree_smote = f1_score(y_test, y_pred_tree_smote)
roc_auc_tree_smote = roc_auc_score(y_test, y_prob_tree_smote)

cm_tree_smote = confusion_matrix(y_test, y_pred_tree_smote)

print('Decision Tree with Grid Search with SMOTE')
print("Best parameters:", grid_tree_smote.best_params_)
print("Accuracy:", accuracy_tree_smote)
print("Precision:", precision_tree_smote)
print("Recall:", recall_tree_smote)
print("F1-score:", f1_tree_smote)
print("ROC-AUC:", roc_auc_tree_smote)

print("\nConfusion Matrix:")
print(cm_tree_smote)

Decision Tree with Grid Search with SMOTE
Best parameters: {'tree__criterion': 'entropy', 'tree__max_depth': None, 'tree__min_samples_leaf': 2, 'tree__min_samples_split': 2}
Accuracy: 0.9866747210574941
Precision: 0.34513274336283184
Recall: 0.6638297872340425
F1-score: 0.45414847161572053
ROC-AUC: 0.8389774090081875

Confusion Matrix:
[[27611   296]
 [   79   156]]


In [2]:
import pandas as pd

tree_1000 = [
    {"Model": "Tree baseline", "Accuracy": 0.99218, "Precision": 0.5316, "Recall": 0.5362, "F1": 0.5339, "ROC-AUC": 0.7661},
    {"Model": "Tree balanced", "Accuracy": 0.99200, "Precision": 0.5243, "Recall": 0.4596, "F1": 0.4898, "ROC-AUC": 0.7280},
    {"Model": "Tree CV", "Accuracy": 0.99220, "Precision": 0.5342, "Recall": 0.5514, "F1": 0.5424, "ROC-AUC": 0.7737},
    {"Model": "Tree tuned", "Accuracy": 0.99332, "Precision": 0.6230, "Recall": 0.5064, "F1": 0.5587, "ROC-AUC": 0.8868},
    {"Model": "Tree + SMOTE", "Accuracy": 0.98667, "Precision": 0.3451, "Recall": 0.6638, "F1": 0.4541, "ROC-AUC": 0.8390}
]

df_tree_1000 = pd.DataFrame(tree_1000)
df_tree_1000

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Tree baseline,0.99218,0.5316,0.5362,0.5339,0.7661
1,Tree balanced,0.99200,0.5243,0.4596,0.4898,0.7280
2,Tree CV,0.99220,0.5342,0.5514,0.5424,0.7737
3,Tree tuned,0.99332,0.6230,0.5064,0.5587,0.8868
4,Tree + SMOTE,0.98667,0.3451,0.6638,0.4541,0.8390


### 1500

### 1 Baseline Decision Tree

In [22]:
X = TW_1500.drop("label", axis=1)
y = TW_1500["label"]

X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,random_state=42, stratify=y)

tree_baseline = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)

y_pred_tree = tree_baseline.predict(X_test)
y_prob_tree = tree_baseline.predict_proba(X_test)[:,1]


accuracy_tree = accuracy_score(y_test, y_pred_tree)
precision_tree = precision_score(y_test, y_pred_tree)
recall_tree = recall_score(y_test, y_pred_tree)
f1_tree = f1_score(y_test, y_pred_tree)
roc_auc_tree = roc_auc_score(y_test, y_prob_tree)

cm_tree = confusion_matrix(y_test, y_pred_tree)

print('1️.Baseline Decision Tree')
print("Accuracy:", accuracy_tree)
print("Precision:", precision_tree)
print("Recall:", recall_tree)
print("F1-score:", f1_tree)
print("ROC-AUC:", roc_auc_tree)

print("\nConfusion Matrix:")
print(cm_tree)

1️.Baseline Decision Tree
Accuracy: 0.9961978537417383
Precision: 0.45544554455445546
Recall: 0.46938775510204084
F1-score: 0.4623115577889447
ROC-AUC: 0.7337132756397382

Confusion Matrix:
[[27989    55]
 [   52    46]]


### ️2.Balanced Decision Tree

In [24]:
tree_balanced = DecisionTreeClassifier(random_state=42,class_weight="balanced").fit(X_train, y_train)

y_pred_tree_bal = tree_balanced.predict(X_test)
y_prob_tree_bal = tree_balanced.predict_proba(X_test)[:,1]

accuracy_tree_bal = accuracy_score(y_test, y_pred_tree_bal)
precision_tree_bal = precision_score(y_test, y_pred_tree_bal)
recall_tree_bal = recall_score(y_test, y_pred_tree_bal)
f1_tree_bal = f1_score(y_test, y_pred_tree_bal)
roc_auc_tree_bal = roc_auc_score(y_test, y_prob_tree_bal)

cm_tree_bal = confusion_matrix(y_test, y_pred_tree_bal)

print('2.Balanced Decision Tree')
print("Accuracy:", accuracy_tree_bal)
print("Precision:", precision_tree_bal)
print("Recall:", recall_tree_bal)
print("F1-score:", f1_tree_bal)
print("ROC-AUC:", roc_auc_tree_bal)

print("\nConfusion Matrix:")
print(cm_tree_bal)

2.Balanced Decision Tree
Accuracy: 0.9964465922819984
Precision: 0.4880952380952381
Recall: 0.41836734693877553
F1-score: 0.45054945054945056
ROC-AUC: 0.7084170210660216

Confusion Matrix:
[[28001    43]
 [   57    41]]


### 3.Decision Tree with Stratified K-Fold Cross-Validation

In [26]:
tree_model = DecisionTreeClassifier(random_state=42)

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results_tree = cross_validate( tree_model,X,y,cv=cv,scoring=scoring)

print('3.Decision Tree with Stratified K-Fold Cross-Validation')
print("Accuracy:", cv_results_tree["test_accuracy"].mean())
print("Precision:", cv_results_tree["test_precision"].mean())
print("Recall:", cv_results_tree["test_recall"].mean())
print("F1-score:", cv_results_tree["test_f1"].mean())
print("ROC-AUC:", cv_results_tree["test_roc_auc"].mean())

3.Decision Tree with Stratified K-Fold Cross-Validation
Accuracy: 0.996894255740717
Precision: 0.5540047857474604
Recall: 0.5471702082895014
F1-score: 0.5497337755683585
ROC-AUC: 0.7728148812114946


### 4.Decision Tree with Grid Search

In [28]:
tree = DecisionTreeClassifier(random_state=42)

param_grid = {
    "max_depth": [3, 5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "criterion": ["gini", "entropy"]
}

grid_tree = GridSearchCV(tree, param_grid,cv=5,scoring="f1", n_jobs=-1).fit(X_train, y_train)
best_tree = grid_tree.best_estimator_


y_pred_tree_grid = best_tree.predict(X_test)
y_prob_tree_grid = best_tree.predict_proba(X_test)[:,1]

accuracy_tree_grid = accuracy_score(y_test, y_pred_tree_grid)
precision_tree_grid = precision_score(y_test, y_pred_tree_grid)
recall_tree_grid = recall_score(y_test, y_pred_tree_grid)
f1_tree_grid = f1_score(y_test, y_pred_tree_grid)
roc_auc_tree_grid = roc_auc_score(y_test, y_prob_tree_grid)

cm_tree_grid = confusion_matrix(y_test, y_pred_tree_grid)

print('4.Decision Tree with Grid Search')
print("Best parameters:", grid_tree.best_params_)
print("Accuracy:", accuracy_tree_grid)
print("Precision:", precision_tree_grid)
print("Recall:", recall_tree_grid)
print("F1-score:", f1_tree_grid)
print("ROC-AUC:", roc_auc_tree_grid)

print("\nConfusion Matrix:")
print(cm_tree_grid)

4.Decision Tree with Grid Search
Best parameters: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5}
Accuracy: 0.9966953308222586
Precision: 0.5274725274725275
Recall: 0.4897959183673469
F1-score: 0.5079365079365079
ROC-AUC: 0.8090866320854402

Confusion Matrix:
[[28001    43]
 [   50    48]]


### 5.Decision Tree with Grid Search with SMOTE

In [30]:
pipeline = Pipeline([("smote", SMOTE(random_state=42)),("tree", DecisionTreeClassifier(random_state=42))])


param_grid = {
    "tree__max_depth": [3, 5, 10, 20, None],
    "tree__min_samples_split": [2, 5, 10],
    "tree__min_samples_leaf": [1, 2, 5],
    "tree__criterion": ["gini", "entropy"]
}

grid_tree_smote = GridSearchCV(pipeline,param_grid,cv=5, scoring="f1",n_jobs=-1).fit(X_train, y_train)


best_tree_smote = grid_tree_smote.best_estimator_
y_pred_tree_smote = best_tree_smote.predict(X_test)
y_prob_tree_smote = best_tree_smote.predict_proba(X_test)[:,1]

accuracy_tree_smote = accuracy_score(y_test, y_pred_tree_smote)
precision_tree_smote = precision_score(y_test, y_pred_tree_smote)
recall_tree_smote = recall_score(y_test, y_pred_tree_smote)
f1_tree_smote = f1_score(y_test, y_pred_tree_smote)
roc_auc_tree_smote = roc_auc_score(y_test, y_prob_tree_smote)

cm_tree_smote = confusion_matrix(y_test, y_pred_tree_smote)

print('Decision Tree with Grid Search with SMOTE')
print("Best parameters:", grid_tree_smote.best_params_)
print("Accuracy:", accuracy_tree_smote)
print("Precision:", precision_tree_smote)
print("Recall:", recall_tree_smote)
print("F1-score:", f1_tree_smote)
print("ROC-AUC:", roc_auc_tree_smote)

print("\nConfusion Matrix:")
print(cm_tree_smote)

Decision Tree with Grid Search with SMOTE
Best parameters: {'tree__criterion': 'entropy', 'tree__max_depth': None, 'tree__min_samples_leaf': 5, 'tree__min_samples_split': 2}
Accuracy: 0.9949186269632577
Precision: 0.38095238095238093
Recall: 0.7346938775510204
F1-score: 0.5017421602787456
ROC-AUC: 0.8754055216438308

Confusion Matrix:
[[27927   117]
 [   26    72]]


In [3]:
import pandas as pd

tree_1500 = [
    {"Model": "Tree baseline", "Accuracy": 0.99620, "Precision": 0.4554, "Recall": 0.4694, "F1": 0.4623, "ROC-AUC": 0.7337},
    {"Model": "Tree balanced", "Accuracy": 0.99645, "Precision": 0.4881, "Recall": 0.4184, "F1": 0.4505, "ROC-AUC": 0.7084},
    {"Model": "Tree CV", "Accuracy": 0.99689, "Precision": 0.5540, "Recall": 0.5472, "F1": 0.5497, "ROC-AUC": 0.7728},
    {"Model": "Tree tuned", "Accuracy": 0.99670, "Precision": 0.5275, "Recall": 0.4898, "F1": 0.5079, "ROC-AUC": 0.8091},
    {"Model": "Tree + SMOTE", "Accuracy": 0.99492, "Precision": 0.3810, "Recall": 0.7347, "F1": 0.5017, "ROC-AUC": 0.8754}
]

df_tree_1500 = pd.DataFrame(tree_1500)
df_tree_1500.sort_values(by="F1", ascending=False).round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
2,Tree CV,0.9969,0.5540,0.5472,0.5497,0.7728
3,Tree tuned,0.9967,0.5275,0.4898,0.5079,0.8091
4,Tree + SMOTE,0.9949,0.3810,0.7347,0.5017,0.8754
0,Tree baseline,0.9962,0.4554,0.4694,0.4623,0.7337
1,Tree balanced,0.9964,0.4881,0.4184,0.4505,0.7084
